## Initial GPT-2 and DistilGPT-2 fine-tuning

This notebook describes initial finetuning carried out on GPT-2 and DistilGPT-2. While the present code only reflects the finetuning of GPT-2, the same pipeline was used for Distilgpt-2.

In [ ]:
! pip install transformers datasets evaluate

In [ ]:
!pip install numpy==1.24

In [ ]:
!pip install accelerate -U

In [3]:
import accelerate
accelerate.__version__

'0.28.0'

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Load dataset

In [5]:
import pandas as pd
from datasets import Dataset

In [ ]:
sents = pd.read_csv('/content/sentences.tsv', sep = '\t')
sents.columns=['text']
sents = Dataset.from_pandas(sents)

In [7]:
sents = sents.train_test_split(test_size=0.2)

In [8]:
sents["train"][0]

{'text': 'They would like to achive as much as they could, but sometimes directly these people are suffering from a huge amount of health problems, deadly desises, mental problems etc.'}

## Preprocess

In [31]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [32]:
tokenizer.pad_token = tokenizer.eos_token

In [33]:
def preprocess_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True)

In [34]:
tokenized_sents = sents.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=sents["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/10745 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2687 [00:00<?, ? examples/s]

In [35]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

In [36]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [37]:
training_args = TrainingArguments(
    output_dir="GPT2_L1_sent_generator",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_sents["train"],
    eval_dataset=tokenized_sents["test"],
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,3.907600,3.723125
2,3.620300,3.624157
3,3.494900,3.597818


TrainOutput(global_step=4032, training_loss=3.705113312554738, metrics={'train_runtime': 1289.7643, 'train_samples_per_second': 24.993, 'train_steps_per_second': 3.126, 'total_flos': 1837463747328000.0, 'train_loss': 3.705113312554738, 'epoch': 3.0})

In [38]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

``trainer.model.push_to_hub`` afterwards, removed for anonymity

## Inference

In [21]:
prompt = "It can be seen"

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=MODEL_PATH) ## removed for anonymity

config.json:   0%|          | 0.00/924 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

In [43]:
generator(prompt)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'It can be seen from this chart that, in 2010 the percentage of people were 61% and 57% in 1995, but when comparing to this year the proportion of aged people in Italy was stable at 56% and 57% in 2000. The percentage'}]